## Importing and Initial Variables

In [107]:
# from todoist_api_python.api import TodoistAPI
import pandas as pd
from utils import credentials as cd, config as config
import requests
from datetime import datetime
from pathlib import Path

COLUMN_TEMPLATE = config.TODOIST_COLUMNS
FILENAME = "my-energysystem-tasks_raw.csv"
mostrecentdate = None
data_path = Path(Path.cwd(),r"data/",FILENAME)

In [4]:
Path.exists(data_path)

False

## Check for existing archive of tasks

In [11]:
# Path.read_bytes(data_path)
if Path.exists(data_path) is False:
    print("file not found.\npull from earliest month start.")
else:
    print("file found. Opening to collect last task date")
    df = pd.read_csv(data_path)
    mostrecentdate = df['completed_at'].max()
    del df

file found. Opening to collect last task date


### Grab the last completed date as a starting point to pull more tasks

In [6]:
if mostrecentdate is None:
    print("No most recent date found")
    start_date = datetime.now()
    start_date = datetime.replace(start_date,day=1)
    end_date = datetime.now()
    
    iso_start_date = start_date.isoformat()
    iso_end_date = end_date.isoformat()
    # datetime.strptime("5/1/2026","%m/%d/%Y")

# end_date = datetime.strptime("5/18/2026","%m/%d/%Y")



No most recent date found


## Prep API call to collect tasks

In [7]:
url = "https://api.todoist.com/api/v1/tasks/completed/by_completion_date/"

headers = {"Authorization": "Bearer " + cd.api_key}

request_body = {
    "since" : iso_start_date,
    "until" : iso_end_date,
    "limit": 200
}

In [8]:
response = requests.get(url, headers=headers,params = request_body)

In [9]:
data = response.json()

items = data.get("items",[])

len(items)

74

## Build Task Table

In [98]:
def dataframeIndexDict(df):
    columns = df.columns
    idx = 0
    column_dict = dict()

    for c in columns.to_list():
        column_dict[idx] = c
        idx += 1
    return column_dict

In [ ]:
task_table = pd.DataFrame(items)

# Save raw copy to reference later

# task_table.to_csv(data_path, index=False)

In [13]:
task_table['completed_at'].max()

'2026-05-22T03:06:39.052900Z'

In [14]:
col = task_table.columns

In [15]:
idx = 0
column_dict = dict()

for v in col.to_list():
    # print(f'adding to dict: Key= {idx} | Value = {v}')
    column_dict[idx] = v
    idx += 1

In [16]:
id_column = (task_table.columns.get_loc("id"), "id")

## Split due date iterable into unique columns

In [ ]:
# slicer = task_table[[id_column[1],'due']]
# slicer = slicer[slicer['due'].notna()]

# duedates = slicer['due'].apply(pd.Series)

# source_names = duedates.columns.tolist()
# updated_names = ["Due" + part.capitalize() for part in source_names]

# rename_zip = zip(source_names, updated_names)
# rename_dict = dict(rename_zip)
# duedates = duedates.rename(rename_dict, axis=1)

# combined_df = slicer.join(duedates)
# duedates = combined_df.iloc[:,[0,2]]
# cleaned_table = pd.merge(left=task_table,right=duedates, on="id")

In [ ]:
def column_is_type(df):
    return df.transform(lambda x: x.apply(type)).drop_duplicates().iloc[0]

def split_column(data,split):
    test = data[[id_column[1],split]]
    test = test[test.iloc[:, 1].notna()]
    # print(column_is_type(test.iloc[:,1]))
    if column_is_type(test.iloc[:,1]) is dict:
        unnest = test[split].apply(pd.Series)
        source_names = unnest.columns.tolist()
        updated_names = [split.capitalize() + part.capitalize() for part in source_names]

        rename_zip = zip(source_names, updated_names)
        rename_dict = dict(rename_zip)

        renamed = unnest.rename(rename_dict, axis=1)
        output = test.join(renamed)
        return output

In [ ]:
date_columns = split_column(task_table,'due')


combined_df = pd.merge(left=task_table, right=date_columns, on=id_column[1],how='left')

## Drop Raw data & Unneeded columns

In [ ]:
final_df = combined_df.drop(columns=COLUMN_TEMPLATE['drop'], errors="ignore")

final_df.info()